In [0]:
import requests
import pandas as pd
from pyspark.sql.functions import current_timestamp

# Part 1: Fetch OpenWeather Secret
print("1. Fetching OpenWeather Secret...")
openweather_key = dbutils.secrets.get(scope="aviation_api_keys", key="openweather_key")

airports = {
    "LIT": {"lat": 34.7294, "lon": -92.2242}, "ATL": {"lat": 33.6407, "lon": -84.4277},
    "DFW": {"lat": 32.8998, "lon": -97.0403}, "ORD": {"lat": 41.9742, "lon": -87.9073},
    "CLT": {"lat": 35.2140, "lon": -80.9431}, "IAH": {"lat": 29.9902, "lon": -95.3368},
    "DEN": {"lat": 39.8617, "lon": -104.6732}, "DAL": {"lat": 32.8459, "lon": -96.8509},
    "STL": {"lat": 38.7480, "lon": -90.3700}, "LGA": {"lat": 40.7772, "lon": -73.8725},
    "DCA": {"lat": 38.8514, "lon": -77.0377}, "LAS": {"lat": 36.0803, "lon": -115.1524},
    "MIA": {"lat": 25.7933, "lon": -80.2906}
}

In [0]:
# Part 2: Get Live Weather Data
weather_data_list = []
weather_url = "https://api.openweathermap.org/data/2.5/weather"

print("2. Fetching Live Weather for 13 routing airports...")
for apt_code, coords in airports.items():
    weather_params = {
        "lat": coords["lat"], 
        "lon": coords["lon"], 
        "appid": openweather_key, 
        "units": "imperial"
    }
    weather_res = requests.get(weather_url, params=weather_params)
    
    if weather_res.status_code == 200:
        data = weather_res.json()
        flattened_weather = {
            "airport_code": apt_code,
            "api_timestamp": data.get("dt"),
            "temperature_f": data.get("main", {}).get("temp"),
            "wind_speed_mph": data.get("wind", {}).get("speed"),
            "weather_condition": data.get("weather", [{}])[0].get("main"),
            "rain_1h": data.get("rain", {}).get("1h", 0.0)
        }
        weather_data_list.append(flattened_weather)
    else:
        print(f"   -> [!] FAILED OpenWeather for {apt_code}: {weather_res.status_code}")

In [0]:
# Part 3: Write to Bronze Delta Table
if weather_data_list:
    print("3. Writing live weather to Bronze layer...")
    pdf_weather = pd.DataFrame(weather_data_list)
    sdf_weather = spark.createDataFrame(pdf_weather)
    sdf_weather = sdf_weather.withColumn("bronze_ingested_at", current_timestamp())
    
    sdf_weather.write.format("delta") \
        .mode("append") \
        .option("mergeSchema", "true") \
        .saveAsTable("aviation_project.bronze_live_weather")
    print(f"   -> SUCCESS: Appended weather for {len(weather_data_list)} airports to aviation_project.bronze_live_weather.")